## Welcome to the Automated Data Science with CrewAI Agents Project

## Project Overview

In this project, I built an end-to-end Agentic Data Science Copilot that helps automate a practical machine learning workflow for predicting supplement product sales.

The system uses weekly supplement sales data from a CSV file, loads the data into a shared pandas DataFrame, and uses a CrewAI team of specialized agents to plan, prepare, model, and evaluate the data science workflow.

The planner agent creates the machine learning strategy, the data analysis and preprocessing agent inspects the dataset and prepares the features, and the modeling and evaluation agent trains a regression model, measures performance, and reports the most important drivers behind the prediction. A custom notebook executor tool allows the agents to generate and run Python code inside the notebook environment, making the workflow interactive and reproducible.

This project demonstrates how multi-agent systems can support real data science work instead of only producing static analysis. The agents collaborate across planning, data preparation, model training, evaluation, and interpretation.

## Business Statement

Businesses often collect sales data, but turning that data into useful forecasts can be slow when teams have to manually inspect the dataset, prepare features, train models, evaluate performance, and explain the results.

This project solves that problem by using a multi-agent AI workflow to automate the core steps of a regression project. The CrewAI agents analyze supplement sales data, prepare it for modeling, train a machine learning model to predict units sold, and summarize model performance and feature importance.

The business value is faster data science execution, more repeatable analysis, clearer model evaluation, and better decision support for sales planning, inventory decisions, and marketing strategy.


In [1]:
# Import necessary libraries

import os
import pandas as pd

from dotenv import load_dotenv
from IPython.display import display, Markdown
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from notebookExecutor import NotebookCodeExecutor, NotebookCodeExecutorSchema

In [2]:
load_dotenv()

# Configure API Key, which is essential for AI agents to use OpenAI models
openai_api_key = os.getenv("OPENAI_API_KEY")

# Initialize the LLM for the agents
llm = LLM(model="openai/gpt-4.1-mini", api_key=openai_api_key)

In [3]:
# Helper function to print markdowns
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))

In [4]:

file_path = "Supplement_Sales_Weekly.csv"
shared_df = pd.read_csv(file_path) # Read the data

In [5]:
shared_df.head()

,Date,Product Name,Category,Units Sold,Price,Discount,Location,Platform
0,2020-01-06,Whey Protein,Protein,161,31.98,0.03,Canada,Walmart
1,2020-01-06,Vitamin C,Vitamin,135,42.51,0.04,UK,Amazon
2,2020-01-06,Fish Oil,Omega,2604,12.91,0.25,Canada,Amazon
3,2020-01-06,Multivitamin,Vitamin,949,16.07,0.08,Canada,Walmart
4,2020-01-06,Pre-Workout,Performance,551,35.47,0.25,Canada,iHerb


Test the Notebook Executor tool i've created 

The tool will 
1.  Accept Python code as a string.
2.  Optionally accept a list of required Python libraries.
3.  Attempt to install the required libraries using pip within the current environment.
4.  Execute the provided Python code directly in the **notebook's global scope** using `exec()`. This allows interaction with existing variables like `shared_df`.
5.  Capture any output printed by the code.
6.  Return the captured output or any error messages (from installation or execution).

**SECURITY WARNING:** This tool executes arbitrary code directly in your environment using `exec(code, globals())`. Please ensure to use Only in trusted environments.

In [6]:
# Let's instantiate the custom tool
notebook_executor_tool = NotebookCodeExecutor(namespace=globals())
print("Custom tool 'NotebookCodeExecutor' instantiated with notebook's global namespace.")

Custom tool 'NotebookCodeExecutor' instantiated with notebook's global namespace.


In [7]:
# Create a simple function to test
def add_numbers(a,b):
    return a + b

In [8]:
# Test the tool
test_code = "print(add_numbers(1,2))"
print("\nTesting tool:\n")
print(notebook_executor_tool.run(code = test_code))

## Tool is working as expected 


Testing tool:

--- Executing Code ---
Code executed successfully. Output:
```output
3

```



### DEFINING THE AGENTS WITH CODE GENERATION FOCUS

In this step, I'll create the CrewAI agents that will work together on the data science workflow. Each agent has a specific role, so the project feels like a small data science team instead of one general assistant trying to do everything.

The planner agent decides the overall machine learning approach. The analysis and preprocessing agent writes Python code to inspect the dataset, clean the data, encode features, and create the train/test split. The modeling and evaluation agent writes Python code to train regression models, evaluate their performance, and explain which features were most important.

The important idea is that these agents are not only describing what should happen. They are asked to generate real Python code and run it with the `notebook_executor_tool`. This allows the notebook to move from planning, to execution, to results in one agent-powered workflow.


In [9]:
# Define the Data Science Planner Agent --> this agent does not need tool (NotebookCodeExecutor)

planner_agent = Agent(role = "Lead Data Scientist and Planner",
                      goal = ("Analyze the objective (predict 'Units Sold') assuming data is in a global pandas DataFrame 'shared_df'. "
                              "Create a step-by-step plan for regression analysis. Instruct subsequent agents on the GOALS for each step."
                              "(e.g., inspect data, preprocess, model, evaluate) and tell them to use the 'Notebook Code Executor' tool "
                              "to WRITE and EXECUTE the necessary Python code."),
                    backstory = ("Experienced data scientist planning ML projects. Knows data is in 'shared_df' and agents will write and execute code using a tool."),
                    llm = llm,
                    allow_delegation = False,
                    verbose = True)

In [10]:
# Define the Data Analysis and Preprocessing Agent (needs access to notebook_executer_tool to generate code)

analyst_preprocessor_agent = Agent(role = "Data Analysis and Preprocessing Expert",
                                   goal = (
        "Follow the plan for data analysis and preprocessing. **Write the necessary Python code** using pandas and scikit-learn "
        "to operate on the global pandas DataFrame 'shared_df'. Your code must perform inspection (shape, info, nulls, describe), "
        "handle date/identifiers (convert 'Date', sort, drop 'Date'/'Product Name'), encode categoricals (OneHotEncode 'Platform' modifying 'shared_df'), "
        "and finally **create the global variables X_train, X_test, y_train, y_test** from 'shared_df' using an 80/20 split (shuffle=False). "
        "Use the 'Notebook Code Executor' tool to execute the code you write. Ensure your generated code includes print statements for key results."),
    
                                   backstory = (
        "Meticulous analyst skilled in writing pandas/sklearn code. Uses the 'Notebook Code Executor' tool to run the generated code. "
        "Knows data is in global 'shared_df' and must create global train/test variables."),
                                   llm = llm,
                                   tools = [notebook_executor_tool],  # Assigning the custom tool explicitly
                                   allow_delegation = False,
                                   verbose = True)


In [11]:
# Define the Modeling and Evaluation Agent (needs access to notebook_executer_tool to generate code)

modeler_evaluator_agent = Agent(role = "Machine Learning Modeler and Evaluator",
                                goal = (
        "Follow the plan for modeling and evaluation. **Write the necessary Python code** using scikit-learn. "
        "Assume global variables X_train, X_test, y_train, y_test exist. Your code must train a RandomForestRegressor(random_state=42), "
        "make predictions on X_test, calculate and print evaluation metrics (MAE, MSE, RMSE, R²), and print the top 10 feature importances. "
        "Use the 'Notebook Code Executor' tool to execute the code you write. "
        "Finally, include the exact Python code you generated and executed in your final response, formatted in a markdown block."
    ),
                                backstory = (
        "ML engineer specialized in regression. Writes scikit-learn code and uses the 'Notebook Code Executor' tool to run it. "
        "Expects global train/test split variables (X_train etc.) to be available."
    ),
    llm = llm,
    tools = [notebook_executor_tool],  # Assigning the custom tool explicitly
    allow_delegation = False,
    verbose = True)


### DEFINING KEY TASKS & RESPONSIBLE AGENT

In this step, I'll define the specific jobs that each CrewAI agent must complete. The agents are the team members, and the tasks are the instructions that tell each team member what to produce.

The planning task asks the planner agent to outline the full machine learning workflow for predicting `Units Sold`. The preprocessing task asks the data agent to inspect `shared_df`, clean the dataset, encode the features, and create `X_train`, `X_test`, `y_train`, and `y_test`. The modeling task asks the model agent to train regression models, evaluate the results, and explain which features are most important.

The key point is that the agents are not just writing explanations. Each task tells the agent to generate Python code and run it with the `Notebook Code Executor` tool. Because the code runs inside the notebook, every agent can use the same shared variables and build on the work completed by the previous task.


In [12]:
# Define the Planning Task

planning_task = Task(
    description = (
        "1. Goal: Create a plan for regression predicting 'Units Sold'.\n"
        "2. Data Context: Global pandas DataFrame 'shared_df' is available.\n"
        "3. Plan Steps: Outline sequence, instructing agents on their GOALS for each step and to use the 'Notebook Code Executor' tool to WRITE and RUN Python code:\n"
        "    a. Goal: Inspect global 'shared_df' (shape, info, nulls, describe).\n"
        "    b. Goal: Preprocess global 'shared_df' (handle Date [to_datetime, sort, drop], drop identifiers ['Product Name'], OneHotEncode 'Platform' [update 'shared_df'], create global X/y vars, create global train/test split vars X_train/test, y_train/test [80/20, shuffle=False]).\n"
        "    c. Goal: Train RandomForestRegressor using global X_train, y_train (use random_state=42).\n"
        "    d. Goal: Evaluate model on global X_test (predict, calc & print MAE, MSE, RMSE, R2).\n"
        "    e. Goal: Extract & print top 10 feature importances from the trained model.\n"
        "5. Output: Numbered plan focusing on the objectives for each data science step."
    ),
    expected_output = (
        "Numbered plan outlining the data science goals for subsequent agents, reminding them to generate code and use the 'Notebook Code Executor' tool, interacting with global variables like 'shared_df' and 'X_train'."
    ),
    agent = planner_agent)


In [13]:
# Define the Data Analysis and Preprocessing Task 

data_analysis_preprocessing_task = Task(
    description = (
        "Follow the analysis/preprocessing plan. Your goal is to inspect and prepare the global 'shared_df' DataFrame and create global training/testing variables. "
        "You MUST **generate Python code** to achieve this and then execute it using the 'Notebook Code Executor' tool. "
        "Specifically, your generated code needs to:\n"
        "1. Inspect the 'shared_df' DataFrame (print shape, info(), isnull().sum(), describe()).\n"
        "2. Convert 'Date' column in 'shared_df' to datetime objects, sort 'shared_df' by 'Date', then drop the 'Date' and 'Product Name' columns from 'shared_df'.\n"
        "3. One-Hot Encode the 'Platform' column in 'shared_df' (use pd.get_dummies, drop_first=True). **Crucially, ensure 'shared_df' DataFrame variable is updated with the result of the encoding.**\n"
        "4. Create a global variable 'y' containing the 'Units Sold' column from 'shared_df'.\n"
        "5. Create a global variable 'X' containing the remaining columns from the updated 'shared_df' (after dropping 'Units Sold').\n"
        "6. Split 'X' and 'y' into global variables: 'X_train', 'X_test', 'y_train', 'y_test' using an 80/20 split with `shuffle=False`. Ensure these four variables are created in the global scope.\n"
        "Make sure your generated code includes necessary imports (like pandas, train_test_split) and print statements for verification (e.g., printing shapes of created variables like X_train.shape)."
        # please note that you will need to  pass the required libraries (e.g., ['pandas', 'scikit-learn']) to the tool if your code uses them
    ),
    expected_output = (
        "Output from the 'Notebook Code Executor' tool showing the successful execution of agent-generated code. This includes printouts confirming:\n"
        "- Initial data inspection results for 'shared_df'.\n"
        "- Confirmation of DataFrame modifications (e.g., shape after encoding).\n"
        "- Confirmation of the creation and shapes of global variables X, y, X_train, X_test, y_train, y_test."
    ),
    agent = analyst_preprocessor_agent,
    tools = [notebook_executor_tool],  
)


In [14]:
# Define the Modeling and Evaluation Task 

modeling_evaluation_task = Task(
    description = (
        "Follow the modeling/evaluation plan. Your goal is to train a model, evaluate it, and report results. "
        "You MUST **generate Python code** assuming global variables X_train, X_test, y_train, y_test exist, and execute it using the 'Notebook Code Executor' tool. "
        "Specifically, your generated code needs to:\n"
        "1. Train a `RandomForestRegressor` model (use `random_state=42`) using the global `X_train` and `y_train` variables. Store the trained model in a global variable named `trained_model`.\n"
        "2. Make predictions on the global `X_test` variable.\n"
        "3. Calculate and print the MAE, MSE, RMSE, and R-squared metrics by comparing predictions against the global `y_test` variable.\n"
        "4. Calculate and print the top 10 feature importances from the trained model (using `X_train.columns` for feature names).\n"
        "Make sure your generated code includes necessary imports (like RandomForestRegressor, metrics functions from sklearn.metrics, numpy, pandas) and print statements for all results.\n"
        "Finally, include the exact Python code you generated and executed within a markdown code block (```python...```) in your final response."
        
    ),
    expected_output = (
        "Output from the 'Notebook Code Executor' tool showing the successful execution of agent-generated code, including:\n"
        "- Printed regression metrics (MAE, MSE, RMSE, R²).\n"
        "- Printed top 10 feature importances.\n"
        "The final response MUST also contain a markdown code block (```python...```) showing the exact Python code that was generated and executed for these steps."
    ),
    agent = modeler_evaluator_agent,
    tools = [notebook_executor_tool],  
)

### CREATING AND RUNNING THE CREW

In this step, I'll bring the agents and tasks together into one CrewAI workflow. The crew is the structure that decides which agents will run, which tasks they will complete, and what order they will follow.

For this project, the crew runs sequentially. That means the planner agent goes first, then the preprocessing agent uses the plan to prepare the data, and finally the modeling agent trains and evaluates the regression models.

When the crew starts, the agents will generate Python code and execute it with the `NotebookCodeExecutor` tool. The verbose output is useful because it lets us watch the agents' reasoning, tool calls, generated code, and final results as the workflow moves from planning to model evaluation.

In [15]:
# Creating the Crew

regression_crew = Crew(
    agents = [planner_agent, analyst_preprocessor_agent, modeler_evaluator_agent],
    tasks = [planning_task, data_analysis_preprocessing_task, modeling_evaluation_task],
    process = Process.sequential,
    verbose = 1,  
    output_log_file = True)

In [16]:
# Kick off the crew execution
print("Starting the Crew execution (Agents will generate code)...")

crew_result = await regression_crew.kickoff_async()

Starting the Crew execution (Agents will generate code)...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 75e496b6-1360-4069-8cfa-310847ba7f19                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Goal: Create a plan for regression predicting 'Units Sold'.                                           │
│  2. Data Context: Global pandas DataFrame 'shared_df' is available.                                             │
│  3. Plan Steps: Outline sequence, instructing agents on their GOALS for each step and to use the 'Notebook      │
│  Code Executor' tool to WRITE and RUN Python code:                                                              │
│      a. Goal: Inspect global 'shared_df' (shape, info, nulls, describe).                                        │
│      b. Goal: Preprocess global 'shared_df' (handle Date [to_datetime, sort, drop], drop identifiers ['Product  │
│  Name'], OneHotEncode 'Platform' [update 'shared_df'], create global X/y vars, create global train/test split   │
│  vars X_train/test, y_train/test [80/20, shuffle=False]).                                                       │
│      c. Goal: Train RandomForestRegressor using global X_train, y_train (use random_state=42).                  │
│      d. Goal: Evaluate model on global X_test (predict, calc & print MAE, MSE, RMSE, R2).                       │
│      e. Goal: Extract & print top 10 feature importances from the trained model.                                │
│  5. Output: Numbered plan focusing on the objectives for each data science step.                                │
│  ID: 0a1820d9-48eb-496d-8ea2-5ff554f91b3e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Data Scientist and Planner                                                                         │
│                                                                                                                 │
│  Task: 1. Goal: Create a plan for regression predicting 'Units Sold'.                                           │
│  2. Data Context: Global pandas DataFrame 'shared_df' is available.                                             │
│  3. Plan Steps: Outline sequence, instructing agents on their GOALS for each step and to use the 'Notebook      │
│  Code Executor' tool to WRITE and RUN Python code:                                                              │
│      a. Goal: Inspect global 'shared_df' (shape, info, nulls, describe).                                        │
│      b. Goal: Preprocess global 'shared_df' (handle Date [to_datetime, sort, drop], drop identifiers ['Product  │
│  Name'], OneHotEncode 'Platform' [update 'shared_df'], create global X/y vars, create global train/test split   │
│  vars X_train/test, y_train/test [80/20, shuffle=False]).                                                       │
│      c. Goal: Train RandomForestRegressor using global X_train, y_train (use random_state=42).                  │
│      d. Goal: Evaluate model on global X_test (predict, calc & print MAE, MSE, RMSE, R2).                       │
│      e. Goal: Extract & print top 10 feature importances from the trained model.                                │
│  5. Output: Numbered plan focusing on the objectives for each data science step.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Machine Learning Modeler and Evaluator                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  from sklearn.ensemble import RandomForestRegressor                                                             │
│  from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score                                  │
│  import numpy as np                                                                                             │
│  import pandas as pd                                                                                            │
│                                                                                                                 │
│  # We found 'Category' and 'Location' columns as categorical string columns, so we must one-hot encode them     │
│  for modeling.                                                                                                  │
│  X_train_encoded = pd.get_dummies(X_train, columns=['Category', 'Location'], drop_first=True)                   │
│  X_test_encoded = pd.get_dummies(X_test, columns=['Category', 'Location'], drop_first=True)                     │
│                                                                                                                 │
│  # Align columns in test set to training set                                                                    │
│  X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)                         │
│                                                                                                                 │
│  # 1. Train RandomForestRegressor                                                                               │
│  trained_model = RandomForestRegressor(random_state=42)                                                         │
│  trained_model.fit(X_train_encoded, y_train)                                                                    │
│                                                                                                                 │
│  # 2. Make predictions on X_test                                                                                │
│  predictions = trained_model.predict(X_test_encoded)                                                            │
│                                                                                                                 │
│  # 3. Calculate and print evaluation metrics                                                                    │
│  mae = mean_absolute_error(y_test, predictions)                                                                 │
│  mse = mean_squared_error(y_test, predictions)                                                                  │
│  rmse = np.sqrt(mse)                                                                                            │
│  r2 = r2_score(y_test, predictions)                                                                             │
│                                                                                                                 │
│  print(f"Mean Absolute Error (MAE): {mae:.4f}")                                                                 │
│  print(f"Mean Squared Error (MSE): {mse:.4f}")                                                                  │
│  print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")  

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Follow the modeling/evaluation plan. Your goal is to train a model, evaluate it, and report results.     │
│  You MUST **generate Python code** assuming global variables X_train, X_test, y_train, y_test exist, and        │
│  execute it using the 'Notebook Code Executor' tool. Specifically, your generated code needs to:                │
│  1. Train a `RandomForestRegressor` model (use `random_state=42`) using the global `X_train` and `y_train`      │
│  variables. Store the trained model in a global variable named `trained_model`.                                 │
│  2. Make predictions on the global `X_test` variable.                                                           │
│  3. Calculate and print the MAE, MSE, RMSE, and R-squared metrics by comparing predictions against the global   │
│  `y_test` variable.                                                                                             │
│  4. Calculate and print the top 10 feature importances from the trained model (using `X_train.columns` for      │
│  feature names).                                                                                                │
│  Make sure your generated code includes necessary imports (like RandomForestRegressor, metrics functions from   │
│  sklearn.metrics, numpy, pandas) and print statements for all results.                                          │
│  Finally, include the exact Python code you generated and executed within a markdown code block                 │
│  (```python...```) in your final response.                                                                      │
│  Agent: Machine Learning Modeler and Evaluator                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 75e496b6-1360-4069-8cfa-310847ba7f19                                                                       │
│  Final Output: ```python                                                                                        │
│  from sklearn.ensemble import RandomForestRegressor                                                             │
│  from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score                                  │
│  import numpy as np                                                                                             │
│  import pandas as pd                                                                                            │
│                                                                                                                 │
│  # We found 'Category' and 'Location' columns as categorical string columns, so we must one-hot encode them     │
│  for modeling.                                                                                                  │
│  X_train_encoded = pd.get_dummies(X_train, columns=['Category', 'Location'], drop_first=True)                   │
│  X_test_encoded = pd.get_dummies(X_test, columns=['Category', 'Location'], drop_first=True)                     │
│                                                                                                                 │
│  # Align columns in test set to training set                                                                    │
│  X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)                         │
│                                                                                                                 │
│  # 1. Train RandomForestRegressor                                                                               │
│  trained_model = RandomForestRegressor(random_state=42)                                                         │
│  trained_model.fit(X_train_encoded, y_train)                                                                    │
│                                                                                                                 │
│  # 2. Make predictions on X_test                                                                                │
│  predictions = trained_model.predict(X_test_encoded)                                                            │
│                                                                                                                 │
│  # 3. Calculate and print evaluation metrics                                                                    │
│  mae = mean_absolute_error(y_test, predictions)                                                                 │
│  mse = mean_squared_error(y_test, predictions)                                                                  │
│  rmse = np.sqrt(mse)                                                                                            │
│  r2 = r2_score(y_test, predictions)                                                                             │
│                                                                                                                 │
│  print(f"Mean Absolute Error (MAE): {mae:.4f}")                                                                 │
│  print(f"Mean Squared Error (MSE): {mse:.4f}")                                                                  │
│  print(f"Root Mean Squared Error (RMSE): {rmse:.4f}") 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [17]:
print("\n\n🏁 Crew execution finished.")
print("\nCrew Final Result (Output of last task):")
print("========================================")

print_markdown(crew_result.raw)



🏁 Crew execution finished.

Crew Final Result (Output of last task):


```python
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# We found 'Category' and 'Location' columns as categorical string columns, so we must one-hot encode them for modeling.
X_train_encoded = pd.get_dummies(X_train, columns=['Category', 'Location'], drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=['Category', 'Location'], drop_first=True)

# Align columns in test set to training set
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# 1. Train RandomForestRegressor
trained_model = RandomForestRegressor(random_state=42)
trained_model.fit(X_train_encoded, y_train)

# 2. Make predictions on X_test
predictions = trained_model.predict(X_test_encoded)

# 3. Calculate and print evaluation metrics
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R2): {r2:.4f}")

# 4. Feature importances
importances = trained_model.feature_importances_
features = X_train_encoded.columns
feature_importance_df = pd.DataFrame({"Feature": features, "Importance": importances})
feature_importance_df_sorted = feature_importance_df.sort_values(by="Importance", ascending=False).head(10)

print("\nTop 10 Feature Importances:")
print(feature_importance_df_sorted.to_string(index=False))
```

Output:
```
Mean Absolute Error (MAE): 160.0538
Mean Squared Error (MSE): 82288.8925
Root Mean Squared Error (RMSE): 286.8604
R-squared (R2): 0.7810

Top 10 Feature Importances:
             Feature  Importance
               Price    0.783860
            Discount    0.110057
    Category_Protein    0.026008
Category_Performance    0.014478
 Category_Fat Burner    0.011808
    Category_Mineral    0.009101
             Walmart    0.008894
         Location_UK    0.007303
               iHerb    0.007224
        Location_USA    0.006216
```

## Task 8: Model Insight and Overview

The model created above shows that the agent workflow was able to move through the core data science process: inspect the dataset, prepare the features, train a regression model, evaluate performance, and explain the most important prediction drivers.

Based on the model output above, the workflow trained and evaluated one model: a `RandomForestRegressor`. The model achieved an R-squared score of about `0.7810`, with an MAE of about `160.05` and an RMSE of about `286.86`. This means the model explained a meaningful amount of the variation in `Units Sold`, while still leaving room for improvement because the prediction errors are not zero.

The feature importance results also give useful business insight. `Price` was the strongest predictor of `Units Sold`, followed by `Discount`. This suggests that product pricing and discount strategy have a strong relationship with sales volume in this dataset. Other features such as product category, location, and platform contributed less compared with price-related features.



Please note that this is for practice and learning. The main goal of this project is to validate that the CrewAI agents can generate code, run it, train one model, and return useful model insights.

For a production level machine learning workflow, it is advisable to train and compare multiple models before selecting the best performing one. Different models can behave differently depending on the data, so comparing metrics such as MAE, RMSE, and R-squared helps avoid choosing a weak model too early. Ultimately, I usually advise using Adjusted R2 instead of R2.

Also, note that, to understand this project properly, you should have some intermediate experience building a machine learning model from scratch. You should be comfortable loading data, selecting a target variable, preparing features, splitting data into train and test sets, training a model, evaluating metrics, and interpreting feature importance. The agents automate parts of the workflow, but the user still needs enough data science knowledge to judge whether the output is correct and useful.

This project is specifically tailored to the supplement sales dataset. We can make it more global to enable userthe notebook users provide dataset path, target column, problem type, date column, and columns to remove. 
